# Comparación de la CNN de referencia y YOLO26n-cls
Juan Fernando Jaimes Pérez y Miguel Andrés Veloza

Dataset de GonzaJL en Kaggle, versión 2. Cuatro clases por imagen, Cyst, Normal, Stone y Tumor.
La partición compartida conserva 10.531 imágenes tras retirar 1.225 copias exactas adicionales.

Selecciona un entorno GPU y ejecuta las celdas en orden. La prueba breve debe finalizar en ambos modelos
antes de iniciar las diez corridas. Los modelos, historiales y resultados se guardan en la carpeta
**Proyecto renal CNN y YOLO** de tu Drive. La cuenta usada debe tener esa carpeta en Mi unidad.
Las corridas completas se conservan al volver a ejecutar el notebook. Una corrida interrumpida se reinicia
con su misma semilla. La prueba se evalúa después de completar los diez entrenamientos.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

drive.mount('/content/drive')
folder = Path('/content/drive/MyDrive/Proyecto renal CNN y YOLO')
assert folder.is_dir(), 'Monta una cuenta que contenga la carpeta del proyecto'
project = Path('/content/renal')
with zipfile.ZipFile(folder / 'proyecto_renal.zip') as archive:
    archive.extractall('/content')
results = folder / 'resultados_t4_shared_v2'
results.mkdir(exist_ok=True)
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv'],text=True))

Mounted at /content/drive
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07



In [2]:
environment = Path('/content/renal_env')
subprocess.run([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(environment)],check=True)
python = str(environment / 'bin/python')
subprocess.run([python,'-m','pip','--version'],check=True)
print('Instalando dependencias; registro visible y guardado en Drive.',flush=True)
with (results / 'installation.log').open('w') as log:
    process = subprocess.Popen([python,'-m','pip','install','-r',str(project / 'requirements.txt')],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout:
        log.write(line)
        log.flush()
        print(line,end='',flush=True)
    status = process.wait()
if status:
    raise RuntimeError(f'La instalación terminó con código {status}. Revisa installation.log.')
process = subprocess.run([python,'-c',"import torch; assert torch.cuda.is_available(); print('GPU comprobada:',torch.__version__,torch.cuda.get_device_name(0))"],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(process.stdout)
process.check_returncode()
subprocess.run([python,str(project / 'src/verify_architecture.py')],check=True)
freeze = subprocess.check_output([python,'-m','pip','freeze'],text=True)
(results / 'dependencies.txt').write_text(freeze)
print('DEPENDENCIAS Y ARQUITECTURA VERIFICADAS',flush=True)


Instalando dependencias; registro visible y guardado en Drive.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 29.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.8/620.8 MB 17.7 MB/s  0:00:24
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 26.6 MB/s  0:00:22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 50.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 58.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 56.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 13.6 MB/s  0:00:25
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 44.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 21.7 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 22.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 11.2 MB/s  0:00:34
   ━━━━━━━━━━━━━━━━━━━━

In [3]:
data = Path('/content/renal_kaggle_v2')
data.mkdir(exist_ok=True)
archive_path = data / 'renal.zip'
expected_hash = '7c4bad073d844775a57a235a3292570726b03603e4f0b8ed43f09e39cf7c09c3'
def digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()

if not archive_path.is_file() or digest(archive_path) != expected_hash:
    temporary_archive = data / 'renal.zip.incomplete'
    with temporary_archive.open('wb') as target:
        for index in range(1,5):
            part = folder / f'renal.zip.part{index:02}'
            assert part.is_file(), f'Falta {part.name}'
            with part.open('rb') as source:
                shutil.copyfileobj(source,target)
    assert digest(temporary_archive) == expected_hash
    temporary_archive.replace(archive_path)
with archive_path.open('rb') as source:
    assert hashlib.file_digest(source,'sha256').hexdigest() == expected_hash
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
    archive.extractall(data / 'raw')
manifest = project / 'results/audit/manifest.csv'
assert hashlib.sha256(manifest.read_bytes()).hexdigest() == 'c2a3b9888bae729b5f2c50289e4d2d8918fa15e699761a20178b7c3d1b77f8dd'
options = ['--backend','shared_torch','--root',str(data / 'raw'),'--manifest',str(manifest),'--output',str(results)]
print('ZIP y manifiesto verificados. Imágenes listas para el experimento.')

ZIP y manifiesto verificados. Imágenes listas para el experimento.


In [ ]:
def run(phase,model,seed,smoke=False):
    command = [python,'-u',str(project / 'src/run.py'),phase,model,str(seed),*options]
    if smoke:
        command.append('--smoke')
    label = 'smoke' if smoke else 'full'
    log_folder = results / 'logs'
    log_folder.mkdir(exist_ok=True)
    print(f'INICIO {label} {model} semilla {seed} {phase}',flush=True)
    with (log_folder / f'{label}-{model}-{seed}-{phase}.log').open('a') as log:
        process = subprocess.Popen(command,cwd=project,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line,end='',flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status,command)
    print(f'FINAL {label} {model} semilla {seed} {phase}',flush=True)

for model in ('cnn','yolo'):
    run('train',model,17,smoke=True)
    run('evaluate',model,17,smoke=True)
print('Las dos pruebas breves finalizaron.')

In [5]:
for phase in ('train','evaluate'):
    for seed in (17,27,37,47,57):
        for model in ('cnn','yolo'):
            run(phase,model,seed)
subprocess.run([python,str(project / 'src/verify_results.py'),'--manifest',str(manifest),'--results',str(results),'--require-checkpoints'],check=True)
process = subprocess.run([python,str(project / 'src/aggregate.py'),str(results),'--excel-fallback'],cwd=project,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(process.stdout)
process.check_returncode()
archive_name = shutil.make_archive(str(folder / 'resultados_t4_shared_v2'),'zip',results)
print('EXPERIMENTO COMPLETO',archive_name)

INICIO full cnn semilla 17 train
Ya entrenado: /content/drive/MyDrive/Proyecto renal CNN y YOLO/resultados_t4_shared_v2/full/cnn-17
FINAL full cnn semilla 17 train
INICIO full yolo semilla 17 train
Ya entrenado: /content/drive/MyDrive/Proyecto renal CNN y YOLO/resultados_t4_shared_v2/full/yolo-17
FINAL full yolo semilla 17 train
INICIO full cnn semilla 27 train
{'epoch': 1, 'train_loss': 1.073331339004825, 'val_loss': 1.0447417497634888, 'val_f1_macro': 0.5013261560499905}
{'epoch': 2, 'train_loss': 0.8194431473202377, 'val_loss': 0.8701416850090027, 'val_f1_macro': 0.6142128676392348}
{'epoch': 3, 'train_loss': 0.713122577316012, 'val_loss': 0.73573237657547, 'val_f1_macro': 0.6945093467791708}
{'epoch': 4, 'train_loss': 0.6078381751518953, 'val_loss': 0.7932726740837097, 'val_f1_macro': 0.6961448447450687}
{'epoch': 5, 'train_loss': 0.550574464378833, 'val_loss': 0.9259613752365112, 'val_f1_macro': 0.6741196033177654}
{'epoch': 6, 'train_loss': 0.4715834645331068, 'val_loss': 1.24746

In [6]:
verification = json.loads((results / 'verification.json').read_text())
assert verification['passed'] and len(verification['runs_checked']) == 10
helper_code = "\"\"\"Registra solo las dependencias transitivas activas del proyecto instalado.\"\"\"\nimport argparse\nfrom collections import defaultdict, deque\nfrom importlib.metadata import distribution\nfrom pathlib import Path\n\nfrom packaging.requirements import Requirement\nfrom packaging.utils import canonicalize_name\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('requirements', type=Path)\n    parser.add_argument('output', type=Path)\n    args = parser.parse_args()\n    extras = defaultdict(set)\n    queue = deque()\n    for line in args.requirements.read_text().splitlines():\n        line = line.strip()\n        if not line or line.startswith('#'):\n            continue\n        requirement = Requirement(line)\n        if requirement.marker and not requirement.marker.evaluate({'extra': ''}):\n            continue\n        name = canonicalize_name(requirement.name)\n        found = distribution(name).version\n        if requirement.specifier and not requirement.specifier.contains(found, prereleases=True):\n            raise RuntimeError(f'Se requiere {requirement}; está instalado {found}.')\n        extras[name].update(requirement.extras)\n        queue.append(name)\n    processed = {}\n    pins = {}\n    while queue:\n        name = queue.popleft()\n        active_extras = frozenset(extras[name])\n        if processed.get(name) == active_extras:\n            continue\n        processed[name] = active_extras\n        installed = distribution(name)\n        pins[name] = installed.version\n        for text in installed.requires or ():\n            requirement = Requirement(text)\n            if requirement.marker and not any(requirement.marker.evaluate({'extra': extra}) for extra in ('', *active_extras)):\n                continue\n            dependency = canonicalize_name(requirement.name)\n            found = distribution(dependency).version\n            if requirement.specifier and not requirement.specifier.contains(found, prereleases=True):\n                raise RuntimeError(f'{name} requiere {requirement}; está instalado {found}.')\n            extras[dependency].update(requirement.extras)\n            queue.append(dependency)\n    args.output.write_text('# Entorno instalado para este sistema y versión de Python.\\n' + '\\n'.join(f'{name}=={pins[name]}' for name in sorted(pins))+'\\n')\n    print(f'Registradas {len(pins)} dependencias activas en {args.output}.')\n\n\nif __name__ == '__main__':\n    main()\n"
process = subprocess.run([python,'-c',helper_code,str(project / 'requirements.txt'),str(results / 'requirements-lock-colab.txt')],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(process.stdout)
process.check_returncode()
shutil.copyfile(project / 'docs/architecture_equivalence.json',results / 'architecture_equivalence.json')
hardware = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv'],text=True)
(results / 'hardware.txt').write_text(hardware)
print(hardware)
archive_name = shutil.make_archive(str(folder / 'resultados_t4_shared_v2'),'zip',results)
print('EXPERIMENTO Y ENTORNO VERIFICADOS',archive_name,flush=True)


Registradas 83 dependencias activas en /content/drive/MyDrive/Proyecto renal CNN y YOLO/resultados_t4_shared_v2/requirements-lock-colab.txt.

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07

EXPERIMENTO Y ENTORNO VERIFICADOS /content/drive/MyDrive/Proyecto renal CNN y YOLO/resultados_t4_shared_v2.zip
